<a href="https://colab.research.google.com/github/Ray23888/Wordle/blob/main/Wordle_Game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import sys

#  1. Dictionary File Creation FOR RUNNING
# This dictionary contains the list of possible secret words.
dictionary_content = """
APPLE
JUICE
TRAIN
PLANE
CRANE
OTHER
WORLD
ADIEU
ARISE
"""
DICTIONARY_FILE = "wordle_dictionary.txt"

# Attempt to create the dictionary file
try:
    with open(DICTIONARY_FILE, "w") as f:
        f.write(dictionary_content.strip())
except Exception as e:
    print(f"Could not create dictionary file: {e}")
    sys.exit(1)


# 2. LetterAnalysis Class (Constants for match types)
class LetterAnalysis:
    """Constants for the result of a letter analysis."""
    EXACT_MATCH = 2
    PARTIAL_MATCH = 1
    NO_MATCH = 0

#  3. analyze function (Core Game Logic)

def analyze(word, target):
    '''
    Checks each letter of the guessed word against the solution word and returns
    a dictionary of indexes with their match type (exact, partial, or none).

    Parameters:
    word (str): The guessed word.
    target (str): The solution word.

    Returns:
    dict: A dictionary mapping index (0-4) to match type.
    '''
    match = LetterAnalysis.EXACT_MATCH
    partial = LetterAnalysis.PARTIAL_MATCH
    no = LetterAnalysis.NO_MATCH

    guess = {}

    for i, v in enumerate(word):
        # Check for Exact Match (Green)
        if v == target[i]:
            guess[i] = match
        # Check for Partial Match (Yellow)
        elif v in target:
            guess[i] = partial
        # No Match (Gray)
        else:
            guess[i] = no

        # Remove the first occurrence of the guessed character in the remaining
        # portion of the solution word. This is necessary for the current logic's function.
        target = target.replace(v, " ", 1)

    return guess


# 4. Statistics Class

class Statistics:
    '''This class calculates all the stats'''
    def __init__(self):
        self.total_games = 0
        self.num_guesses_list = []
        self.games_lost = 0
        self.games_won = 0
        self.win_percentage = 0
        self.avg_guesses = 0

        # Initialize dictionary to track guess distribution
        self.guess_dictionary = {i: 0.0 for i in range(1, 7)}

    def add_lost_game(self):
        '''Record that a game was played but not won'''
        self.total_games += 1
        self.games_lost += 1

    def add_won_game(self, num_guesses):
        '''Record that a game was played and won'''
        self.total_games += 1
        self.games_won += 1

        self.num_guesses_list.append(num_guesses)

    def get_guess_distribution(self):
        '''Calculates and normalizes the distribution of guesses needed for wins'''

        # Calculate correct counts first
        temp_counts = {i: 0 for i in range(1, 7)}
        for i in self.num_guesses_list:
            if i <= 6:
                temp_counts[i] += 1

        # Now normalize to percentages/ratios
        if self.total_games > 0:
            for key in self.guess_dictionary:
                self.guess_dictionary[key] = temp_counts.get(key, 0) / self.total_games
        else:
            self.guess_dictionary = {i: 0.0 for i in range(1, 7)}

        return self.guess_dictionary

    def get_success_metrics(self):
        '''Calculates win percentage and average guesses per win'''
        self.win_percentage = self.games_won / self.total_games if self.total_games > 0 else 0
        self.avg_guesses = sum(self.num_guesses_list) / self.games_won if self.games_won > 0 else 0

        return self.win_percentage, self.avg_guesses

    def print_stats(self):
        # Print end-of-game statistics
        print("\n" + "="*30)
        print(f'{"Statistics":^30}')
        print("="*30)

        print(f'{"% Won: "}{(self.win_percentage * 100):.0f}% | {"Average Solve: "}{self.avg_guesses:.1f}')

        print("\n" + f'{"Guess Distribution":^30}')
        for i in range(1, 7):
            print(f'{i} Guesses: {self.guess_dictionary[i] * 100:.0f}%')

        percent_lost = (self.games_lost / self.total_games) * 100 if self.total_games > 0 else 0
        print(f'N/S (Lost): {percent_lost:.0f}%')
        print("="*30)


# 5. colored_word function
def colored_word(word, analysis_result):
    """
    Colors the word output using ANSI escape codes for Green, Yellow, and Gray
    backgrounds with Bold Black text for maximum visibility.
    """
    # ANSI escape codes
    GREEN_BG = "\x1b[42m"  # Green Background (Exact match)
    YELLOW_BG = "\x1b[43m" # Yellow Background (Partial match)
    GRAY_BG = "\x1b[47m"   # Light Gray Background (No match)
    BLACK_TEXT_BOLD = "\x1b[30;1m" # Black Text, Bold
    RESET = "\x1b[0m"      # Reset all formatting

    output = []

    color_map = {
        LetterAnalysis.EXACT_MATCH: GREEN_BG,
        LetterAnalysis.PARTIAL_MATCH: YELLOW_BG,
        LetterAnalysis.NO_MATCH: GRAY_BG,
    }

    for i, char in enumerate(word):
        match_type = analysis_result.get(i, LetterAnalysis.NO_MATCH)
        bg_code = color_map.get(match_type, RESET)


        output.append(f"{bg_code}{BLACK_TEXT_BOLD} {char} {RESET}")

    return "".join(output)


#  6. run_game function (Main Logic)

def run_game(file_name):
    '''
    Start a game by using the random module to select a secret word and manage the game loop.
    '''
    stats1 = Statistics()

    # Load the dictionary (only to define possible secret words)
    try:
        with open(file_name,"r") as f:
            wrd1 = [word.strip().upper() for word in f.read().split() if len(word.strip()) == 5]
    except FileNotFoundError:
        print(f"Error: Dictionary file '{file_name}' not found.")
        return

    if not wrd1:
        print("Error: Dictionary is empty or contains no 5-letter words.")
        return

    # Select a random word from the list
    Secret_word = random.choice(wrd1).upper()

    num_guesses = 1
    game_status = "Unknown"

    print("\n--- Wordle Game Starting ---")
    print("Guess any 5-letter word. You have 6 attempts.")

    while num_guesses <= 6:
        try:
            user_input = input(f'Enter Guess {num_guesses} (5 letters): ').upper()
        except EOFError:
            print('\nGame Interrupted.')
            stats1.add_lost_game()
            game_status = 'lost'
            break

        # Validation: Only checks for 5 letters
        if len(user_input) != 5:
            print('invalid guess (must be exactly 5 letters long)')
            continue

        # Process the guess
        ana = analyze(user_input, Secret_word)
        print(colored_word(user_input, ana))

        # Check if the user won
        if user_input == Secret_word:
            print('\nWINNER! Congratulations!')
            game_status = 'success won'
            stats1.add_won_game(num_guesses)
            break

        num_guesses += 1 # Increment only after a valid, incorrect guess

    # Check if the loop finished because the user lost
    if num_guesses > 6 and game_status == "Unknown":
        print(f'\nSORRY YOU LOST. The secret word was: {Secret_word}')
        game_status = 'lost'
        stats1.add_lost_game()

    # Final stats printout
    stats1.get_guess_distribution()
    stats1.get_success_metrics()
    stats1.print_stats()

    # Play again loop
    while True:
        try:
            user_input = input("Enter 'y' to play again or 'n' to quit: ").lower()
        except EOFError:
            break

        if user_input == 'y':
            run_game(file_name)
            break
        elif user_input == 'n':
            print("Thanks for playing!")
            break
        else:
            print("Invalid input. Please enter 'y' or 'n'.")


#  7. Main Execution Block

if __name__ == '__main__':
    run_game(DICTIONARY_FILE)


--- Wordle Game Starting ---
Guess any 5-letter word. You have 6 attempts.
Enter Guess 1 (5 letters): APUUL
 A  P  U  U  L 
Enter Guess 2 (5 letters): APPLE
 A  P  P  L  E 
Enter Guess 3 (5 letters): RIGHT
 R  I  G  H  T 
Enter Guess 4 (5 letters): JUICE
 J  U  I  C  E 
Enter Guess 5 (5 letters): CRANE
 C  R  A  N  E 
Enter Guess 6 (5 letters): PLANE
 P  L  A  N  E 

WINNER! Congratulations!

          Statistics          
% Won: 100% | Average Solve: 6.0

      Guess Distribution      
1 Guesses: 0%
2 Guesses: 0%
3 Guesses: 0%
4 Guesses: 0%
5 Guesses: 0%
6 Guesses: 100%
N/S (Lost): 0%
